In [1]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import MultSKAN components
from MultSKAN import MultSKANNetwork, larctan

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# MultSKAN Model for MNIST
class MultSKAN_MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.model = MultSKANNetwork(
            n_a_list=[784, 256, 128, 10],
            n_m_list=[0, 64, 32, 0],
            basis_function=larctan,
            bias=True,
            device=device
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.model(x)
        return x

# MultSKAN Model for CIFAR10
class MultSKAN_CIFAR10(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.model = MultSKANNetwork(
            n_a_list=[3072, 512, 256, 10],
            n_m_list=[0, 128, 64, 0],
            basis_function=larctan,
            bias=True,
            device=device
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.model(x)
        return x

# Training function
def train(model, train_loader, optimizer, criterion, epoch):
    model.train()
    total_loss = 0
    correct = 0
    for batch_idx, (data, target) in tqdm(enumerate(train_loader)):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()

    print(f"Train Epoch: {epoch} \tLoss: {total_loss/len(train_loader):.4f} \tAccuracy: {100. * correct / len(train_loader.dataset):.2f}%")

# Test function
def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()

    test_loss /= len(test_loader)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')

# Data loaders
def get_mnist_loaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST('./data', train=False, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

def get_cifar10_loaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

# Choose dataset and model
dataset = 'MNIST'  # or 'CIFAR10'

if dataset == 'MNIST':
    train_loader, test_loader = get_mnist_loaders()
    model = MultSKAN_MNIST().to(device)
elif dataset == 'CIFAR10':
    train_loader, test_loader = get_cifar10_loaders()
    model = MultSKAN_CIFAR10().to(device)
else:
    raise ValueError('Unknown dataset')

# Optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop
n_epochs = 10
for epoch in range(1, n_epochs + 1):
    train(model, train_loader, optimizer, criterion, epoch)
    test(model, test_loader, criterion)


Using device: cuda


469it [00:06, 73.75it/s]


Train Epoch: 1 	Loss: 0.2565 	Accuracy: 92.32%

Test set: Average loss: 0.1553, Accuracy: 9510/10000 (95.10%)



469it [00:06, 74.21it/s]


Train Epoch: 2 	Loss: 0.1074 	Accuracy: 96.66%

Test set: Average loss: 0.1000, Accuracy: 9680/10000 (96.80%)



...

Train Epoch: 8 	Loss: 0.0366 	Accuracy: 98.83%

Test set: Average loss: 0.0802, Accuracy: 9766/10000 (97.66%)



469it [00:06, 75.92it/s]


Train Epoch: 9 	Loss: 0.0349 	Accuracy: 98.89%

Test set: Average loss: 0.0747, Accuracy: 9771/10000 (97.71%)



469it [00:06, 76.37it/s]


Train Epoch: 10 	Loss: 0.0290 	Accuracy: 99.03%

Test set: Average loss: 0.0902, Accuracy: 9732/10000 (97.32%)



In [2]:
# Choose dataset and model
dataset = 'CIFAR10'  # or 'CIFAR10'

if dataset == 'MNIST':
    train_loader, test_loader = get_mnist_loaders()
    model = MultSKAN_MNIST().to(device)
elif dataset == 'CIFAR10':
    train_loader, test_loader = get_cifar10_loaders()
    model = MultSKAN_CIFAR10().to(device)
else:
    raise ValueError('Unknown dataset')

# Optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop
n_epochs = 10
for epoch in range(1, n_epochs + 1):
    train(model, train_loader, optimizer, criterion, epoch)
    test(model, test_loader, criterion)

391it [00:11, 32.76it/s]


Train Epoch: 1 	Loss: 1.6373 	Accuracy: 41.53%

Test set: Average loss: 1.5476, Accuracy: 4475/10000 (44.75%)



391it [00:11, 32.78it/s]


Train Epoch: 2 	Loss: 1.4741 	Accuracy: 47.50%

Test set: Average loss: 1.4893, Accuracy: 4657/10000 (46.57%)



391it [00:11, 32.79it/s]


...

Train Epoch: 8 	Loss: 1.1692 	Accuracy: 58.72%

Test set: Average loss: 1.4239, Accuracy: 5046/10000 (50.46%)



391it [00:11, 32.64it/s]


Train Epoch: 9 	Loss: 1.1202 	Accuracy: 60.47%

Test set: Average loss: 1.4472, Accuracy: 5037/10000 (50.37%)



391it [00:11, 32.70it/s]


Train Epoch: 10 	Loss: 1.0721 	Accuracy: 62.38%

Test set: Average loss: 1.4448, Accuracy: 5089/10000 (50.89%)

